In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

import joblib

In [2]:
df = pd.read_csv("cleaned_data.csv")

In [3]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0,yes,southwest,16884.92400
1,18.0,male,33.770,1,no,southeast,1725.55230
2,28.0,male,33.000,3,no,southeast,4449.46200
3,33.0,male,22.705,0,no,northwest,21984.47061
4,32.0,male,28.880,0,no,northwest,3866.85520


In [4]:
X = df.drop("charges", axis=1)
y_reg = df["charges"]
y_clf = (y_reg > y_reg.median()).astype(int)

In [5]:
X = pd.get_dummies(X,drop_first=True)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X,y_clf,test_size=0.2,random_state=42,stratify=y_clf)

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

1) Decision Tree Baseline.

In [8]:
dt_baseline = DecisionTreeClassifier(random_state=42)

In [9]:
dt_baseline.fit(X_train_scaled,y_train)

DecisionTreeClassifier(random_state=42)

In [10]:
train_predictions = dt_baseline.predict(X_train_scaled)

In [11]:
test_predictions = dt_baseline.predict(X_test_scaled)

In [12]:
train_accuracy = accuracy_score(y_train,train_predictions)

print("Training Accuracy :", train_accuracy)

Training Accuracy : 0.9990645463049579


In [13]:
test_accuracy = accuracy_score(y_test,test_predictions)

print("Test Accuracy :", test_accuracy)

Test Accuracy : 0.8880597014925373


In [14]:
print("Training Accuracy :", round(train_accuracy,4))
print("Testing Accuracy  :", round(test_accuracy,4))

Training Accuracy : 0.9991
Testing Accuracy  : 0.8881


2. Controlled Decision Tree.

In [15]:
controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

In [16]:
controlled_tree.fit(X_train_scaled,y_train)

DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)

In [17]:
controlled_train_pred = controlled_tree.predict(X_train_scaled)

In [18]:
controlled_test_pred = controlled_tree.predict(X_test_scaled)

In [19]:
controlled_train_accuracy = accuracy_score(y_train,controlled_train_pred)

print("Controlled Tree Training Accuracy:", controlled_train_accuracy)

Controlled Tree Training Accuracy: 0.930776426566885


In [20]:
controlled_test_accuracy = accuracy_score(y_test,controlled_test_pred)

print("Controlled Tree Testing Accuracy:", controlled_test_accuracy)

Controlled Tree Testing Accuracy: 0.9253731343283582


In [21]:
comparison_tree = pd.DataFrame({
    "Model": ["Baseline Decision Tree","Controlled Decision Tree"],
    "Training Accuracy": [train_accuracy,controlled_train_accuracy],
    "Testing Accuracy": [test_accuracy,controlled_test_accuracy]
})

comparison_tree

,Model,Training Accuracy,Testing Accuracy
0,Baseline Decision Tree,0.999065,0.888060
1,Controlled Decision Tree,0.930776,0.925373


3. Gini Tree

In [28]:
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train)

DecisionTreeClassifier(max_depth=5, random_state=42)

In [29]:
gini_predictions = gini_tree.predict(X_test_scaled)
gini_accuracy = accuracy_score(y_test,gini_predictions)

print("Gini Accuracy:", gini_accuracy)

Gini Accuracy: 0.9216417910447762


4. Entropy Tree

In [30]:
entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)

In [31]:
entropy_predictions = entropy_tree.predict(X_test_scaled)
entropy_accuracy = accuracy_score(y_test,entropy_predictions)

print("Entropy Accuracy:", entropy_accuracy)

Entropy Accuracy: 0.914179104477612


In [33]:
comparison = pd.DataFrame({
    "Criterion": ["Gini", "Entropy"],
    "Test Accuracy": [gini_accuracy,entropy_accuracy]
})

comparison

,Criterion,Test Accuracy
0,Gini,0.921642
1,Entropy,0.914179


5. Random Forest

In [34]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

In [35]:
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(max_depth=10, random_state=42)

In [36]:
rf_train_pred = rf_model.predict(X_train_scaled)

In [37]:
rf_test_pred = rf_model.predict(X_test_scaled)

In [38]:
rf_train_accuracy = accuracy_score(y_train,rf_train_pred)

print("Training Accuracy:", rf_train_accuracy)

Training Accuracy: 0.9625818521983162


In [39]:
rf_test_accuracy = accuracy_score(y_test,rf_test_pred)

print("Testing Accuracy:", rf_test_accuracy)

Testing Accuracy: 0.9291044776119403


In [40]:
rf_prob = rf_model.predict_proba(X_test_scaled)[:,1]
rf_auc = roc_auc_score(y_test,rf_prob)

print("Random Forest AUC:", rf_auc)

Random Forest AUC: 0.9558921808866117


In [41]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})
importance = importance.sort_values(by="Importance",ascending=False)

importance.head(5)

,Feature,Importance
0,age,0.507526
4,smoker_yes,0.296341
1,bmi,0.111506
2,children,0.042990
3,sex_male,0.014120


In [42]:
top5 = importance.head(5)
top5

,Feature,Importance
0,age,0.507526
4,smoker_yes,0.296341
1,bmi,0.111506
2,children,0.042990
3,sex_male,0.014120


6. Gradient Boosting Model

In [44]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

In [45]:
gb_model.fit(X_train_scaled, y_train)

GradientBoostingClassifier(random_state=42)

In [46]:
gb_train_pred = gb_model.predict(X_train_scaled)
gb_test_pred = gb_model.predict(X_test_scaled)

In [47]:
gb_train_accuracy = accuracy_score(y_train,gb_train_pred)

print("Training Accuracy:", gb_train_accuracy)

Training Accuracy: 0.9522918615528532


In [48]:
gb_test_accuracy = accuracy_score(y_test,gb_test_pred)

print("Testing Accuracy:", gb_test_accuracy)

Testing Accuracy: 0.9291044776119403


In [49]:
gb_prob = gb_model.predict_proba(X_test_scaled)[:,1]
gb_auc = roc_auc_score(y_test,gb_prob)

print("Gradient Boosting AUC:", gb_auc)

Gradient Boosting AUC: 0.9459790599242593


7. Feature Ablation Study

In [50]:
least5 = importance.sort_values(by="Importance",ascending=True).head(5)
least5

,Feature,Importance
5,region_northwest,0.008134
7,region_southwest,0.009402
6,region_southeast,0.009980
3,sex_male,0.014120
2,children,0.042990


In [51]:
least5_features = least5["Feature"].tolist()
print(least5_features)

['region_northwest', 'region_southwest', 'region_southeast', 'sex_male', 'children']


In [52]:
X_train_reduced = X_train.drop(columns=least5_features)
X_test_reduced = X_test.drop(columns=least5_features)

In [53]:
scaler2 = StandardScaler()
X_train_reduced_scaled = scaler2.fit_transform(X_train_reduced)
X_test_reduced_scaled = scaler2.transform(X_test_reduced)

In [54]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
rf_reduced.fit(X_train_reduced_scaled,y_train)

RandomForestClassifier(max_depth=10, random_state=42)

In [55]:
reduced_prob = rf_reduced.predict_proba(X_test_reduced_scaled)[:,1]
reduced_auc = roc_auc_score(y_test,reduced_prob)

print("Reduced Model AUC:", reduced_auc)

Reduced Model AUC: 0.9296057028291379


In [56]:
ablation = pd.DataFrame({
    "Model":["Full Model","Reduced Model"],
    "AUC":[rf_auc,reduced_auc]
})
ablation

,Model,AUC
0,Full Model,0.955892
1,Reduced Model,0.929606


In [57]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [59]:
logistic_cv = LogisticRegression(max_iter=1000)

In [60]:
logistic_scores = cross_val_score(
    logistic_cv,
    X,
    y_clf,
    cv=cv,
    scoring="roc_auc"
)

In [61]:
tree_scores = cross_val_score(
    controlled_tree,
    X,
    y_clf,
    cv=cv,
    scoring="roc_auc"
)

In [62]:
rf_scores = cross_val_score(
    rf_model,
    X,
    y_clf,
    cv=cv,
    scoring="roc_auc"
)

In [63]:
gb_scores = cross_val_score(
    gb_model,
    X,
    y_clf,
    cv=cv,
    scoring="roc_auc"
)

In [64]:
cv_results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Controlled Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Mean AUC":[
        logistic_scores.mean(),
        tree_scores.mean(),
        rf_scores.mean(),
        gb_scores.mean()
    ],
    "Std AUC":[
        logistic_scores.std(),
        tree_scores.std(),
        rf_scores.std(),
        gb_scores.std()
    ]
})
cv_results

,Model,Mean AUC,Std AUC
0,Logistic Regression,0.945816,0.015927
1,Controlled Decision Tree,0.933051,0.021530
2,Random Forest,0.949711,0.016761
3,Gradient Boosting,0.949669,0.016917


GridSearchCV

In [65]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

In [66]:
param_grid = {
    "randomforestclassifier__n_estimators":[50,100,200],
    "randomforestclassifier__max_depth":[5,10,None],
    "randomforestclassifier__min_samples_leaf":[1,5]
}

In [67]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring="roc_auc",
    n_jobs=-1
)

In [68]:
grid_search.fit(X_train,y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('simpleimputer',
                                        SimpleImputer(strategy='median')),
                                       ('standardscaler', StandardScaler()),
                                       ('randomforestclassifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'randomforestclassifier__max_depth': [5, 10, None],
                         'randomforestclassifier__min_samples_leaf': [1, 5],
                         'randomforestclassifier__n_estimators': [50, 100,
                                                                  200]},
             scoring='roc_auc')

In [69]:
print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 100}


In [70]:
print("Best CV Score:")
print(grid_search.best_score_)

Best CV Score:
0.9483235233224315


In [71]:
best_pipeline = grid_search.best_estimator_

Manual Learning Curve

In [72]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

In [73]:
learning_curve = []

In [74]:
for f in fractions:
    size = int(f * len(X_train))
    X_subset = X_train.iloc[:size]
    y_subset = y_train.iloc[:size]
    best_pipeline.fit(X_subset,y_subset)
    train_prob = best_pipeline.predict_proba(X_subset)[:,1]
    train_auc = roc_auc_score(y_subset,train_prob)
    test_prob = best_pipeline.predict_proba(X_test)[:,1]
    test_auc = roc_auc_score(y_test,test_prob)
    learning_curve.append([f,train_auc,test_auc])

In [75]:
learning_curve_table = pd.DataFrame(learning_curve,
    columns=["Training Fraction","Training AUC","Test AUC"]
)
learning_curve_table

,Training Fraction,Training AUC,Test AUC
0,0.2,1.000000,0.937737
1,0.4,1.000000,0.933616
2,0.6,0.999961,0.936734
3,0.8,0.999839,0.952161
4,1.0,0.999690,0.955892


In [76]:
joblib.dump(best_pipeline,"best_model.pkl")

['best_model.pkl']

In [78]:
sample = pd.DataFrame({
    "age":[25,55],
    "bmi":[24.5,33.2],
    "children":[0,2],
    "sex_male":[0,1],
    "smoker_yes":[0,1],
    "region_northwest":[0,1],
    "region_southeast":[1,0],
    "region_southwest":[0,0]
})

In [82]:
print(X_train.columns.tolist())

['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']
